In [35]:
import pandas as pd
import numpy as np
import sklearn

In [36]:
df = pd.read_csv(r'C:\Users\user\Desktop\Skilfactory\data1\netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,"August 14, 2020",2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,"November 16, 2017",2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi..."
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,"January 1, 2020",2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...


In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [38]:
model = TfidfVectorizer(stop_words='english')

In [39]:
df['description'] = df['description'].fillna('')

In [40]:
feature_matrix = model.fit_transform(df['description'])

In [41]:
feature_matrix.shape

(7787, 17905)

In [42]:
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(feature_matrix, feature_matrix)

In [43]:
indices = pd.Series(df.index,index=df['title']).drop_duplicates()

In [44]:
def get_recommendations(title):
    idx = indices[title]
    #вычисляем попарные коэффициенты косинусной близости
    scores = list(enumerate(cosine_sim[idx]))
    #сортируем фильмы на основании коэффициентов косинусной близости по убыванию
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    #выбираем десять наибольших значений косинусной близости; нулевую не берём, т. к. это тот же фильм
    scores =   scores[1:11]
    #забираем индексы
    ind_movie = [i[0] for i in scores]
    #возвращаем названия по индексам
    return df['title'].iloc[ind_movie]

In [45]:
get_recommendations("Balto")

709                Balto 2: Wolf Quest
7446                           Vroomiz
1338    Chilling Adventures of Sabrina
7388                          Vampires
1770                          Dinotrux
2767                     Hold the Dark
5540                 Shanghai Fortress
4041                             Mercy
2582                       Half & Half
1365        Christmas in the Heartland
Name: title, dtype: str

In [46]:
!pip install scikit-surprise


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
from surprise import Dataset
from surprise import Reader
from surprise.dataset import BUILTIN_DATASETS #с помощью данного объекта мы можем использовать встроенные датасеты


In [48]:
from surprise import Dataset
from surprise import Reader

# Используем встроенный датасет MovieLens-100k
data = Dataset.load_builtin('ml-100k')

print("✅ Встроенный датасет загружен!")
print(f"Тип данных: {type(data)}")

✅ Встроенный датасет загружен!
Тип данных: <class 'surprise.dataset.DatasetAutoFolds'>


In [49]:
df = pd.DataFrame(data.raw_ratings, columns=['userId', 'movieId', 'rating', 'timestamp'])

In [50]:
df['movieId'].nunique()

1682

In [51]:
df['userId'].nunique()

943

In [52]:
df['rating'].value_counts()

rating
4.0    34174
3.0    27145
5.0    21201
2.0    11370
1.0     6110
Name: count, dtype: int64

In [53]:
from surprise.model_selection import train_test_split
import surprise

In [54]:
df

,userId,movieId,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596
...,...,...,...,...
99995,880,476,3.0,880175444
99996,716,204,5.0,879795543
99997,276,1090,1.0,874795795
99998,13,225,2.0,882399156


In [63]:
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)

In [64]:
from surprise.model_selection import train_test_split
trainset, testset = train_test_split(data, test_size=0.25, random_state=13)
len(testset)

25000

In [65]:
from surprise import SVD, KNNBasic, accuracy

In [66]:
sim_options = {
    'name': 'cosine',
    'user_based': False
}
 
knn = KNNBasic(sim_options=sim_options)

In [67]:
knn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [70]:
predictions = knn.test(testset)


In [75]:
for pred in predictions:
    if pred.uid == '500' and pred.iid == '699':
        print(pred.r_ui)
        print(pred.est)

3.0
3.4747900407148165


In [76]:
accuracy.rmse(predictions)

RMSE: 1.0272


1.0271678039029761

In [78]:
pred = pd.DataFrame(predictions)
pred.sort_values(by=['est'],inplace=True,ascending = False)
pred

,uid,iid,r_ui,est,details
22469,849,234,5.0,4.951929,"{'actual_k': 19, 'was_impossible': False}"
1974,849,427,4.0,4.950547,"{'actual_k': 19, 'was_impossible': False}"
8272,849,568,4.0,4.949215,"{'actual_k': 19, 'was_impossible': False}"
5138,849,174,5.0,4.947691,"{'actual_k': 19, 'was_impossible': False}"
22021,688,1127,5.0,4.928412,"{'actual_k': 15, 'was_impossible': False}"
...,...,...,...,...,...
15746,405,194,1.0,1.000000,"{'actual_k': 40, 'was_impossible': False}"
21245,405,197,4.0,1.000000,"{'actual_k': 40, 'was_impossible': False}"
13891,405,511,2.0,1.000000,"{'actual_k': 40, 'was_impossible': False}"
21639,181,151,2.0,1.000000,"{'actual_k': 40, 'was_impossible': False}"


In [80]:
recom = pred[pred.uid =='849']['iid'].to_list()
recom

['234', '427', '568', '174']

In [81]:
sim_options = {
    'name': 'cosine',
    'user_based': True
}
knn = KNNBasic(sim_options=sim_options)
knn.fit(trainset)
predictions = knn.test(testset)
accuracy.rmse(predictions)

Computing the cosine similarity matrix...
Done computing similarity matrix.
RMSE: 1.0175


1.0174852296380237

In [82]:
model = SVD()
model.fit(trainset)
pred = model.test(testset)
accuracy.rmse(pred)

RMSE: 0.9415


0.9414506034074566